<a href="https://colab.research.google.com/github/miku-main/SignBridge/blob/main/ml/signbridge_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SignBridge: ML training notebook (letters)
Run the cells top to bottom. Everything important is saved to Google Drive so a disconnect doesn't wipe your work.

**Steps:** 1 Drive, 2 install, 3 normalize.py, 4 Kaggle data, 5 extract landmarks, 6 baseline, 7 train MLP, 8 confusion matrix, 9 export, 10 sanity check.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
BASE = '/content/drive/MyDrive/signbridge'
os.makedirs(BASE, exist_ok=True)
print('Saving to', BASE)

## 2. Install pinned versions
If `mp.solutions` is missing later, this pin is why it works. Restart the runtime if Colab asks.

In [ ]:
!pip install -q mediapipe opencv-python-headless
!wget -q -O /content/hand_landmarker.task https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task

## 3. Shared `normalize()`
This is the contract with your frontend teammate. It is saved to Drive and should also be committed to the GitHub repo. Port it to JS and compare outputs on the test pair below.

In [ ]:
%%writefile /content/drive/MyDrive/signbridge/normalize.py
import numpy as np

def normalize(landmarks):
    # landmarks: (21, 3) MediaPipe x, y, z. Returns a 63-length vector.
    lm = np.array(landmarks, dtype=np.float32)
    lm = lm - lm[0]                    # wrist becomes the origin
    scale = np.linalg.norm(lm[9])      # wrist to middle-finger base
    lm = lm / (scale + 1e-6)
    return lm.flatten()

In [ ]:
import sys, numpy as np
sys.path.insert(0, BASE)
from normalize import normalize

# Test pair to send to your frontend teammate (their JS must produce the same numbers)
rng = np.random.RandomState(0)
test_in = rng.rand(21, 3)
test_out = normalize(test_in)
print('input first 6 :', test_in.flatten()[:6].round(4).tolist())
print('output first 6:', test_out[:6].round(4).tolist())

## 4. Get the Kaggle ASL Alphabet dataset
Create an API token at kaggle.com, Settings, API, Create New Token. Upload the `kaggle.json` file when prompted.

In [ ]:
import os
from google.colab import userdata

os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')

!pip install -q -U kaggle
!kaggle datasets download -d grassknoted/asl-alphabet -p /content/data --unzip -q

In [ ]:
import glob
roots = sorted(glob.glob('/content/data/**/asl_alphabet_train', recursive=True), key=len)
print(roots)
ROOT = roots[-1]          # the deepest match holds the letter folders
print('Using', ROOT, os.listdir(ROOT)[:10])

## 5. Extract landmarks (saved to Drive)
Capped at 300 images per letter to keep it fast. Skips non-letter folders. Watch the per-letter counts: low counts mean MediaPipe fails to detect that sign.

In [ ]:
import cv2, pandas as pd, mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision
from tqdm import tqdm

options = vision.HandLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path='/content/hand_landmarker.task'),
    running_mode=vision.RunningMode.IMAGE,
    num_hands=1,
    min_hand_detection_confidence=0.2,
    min_hand_presence_confidence=0.2,
)
detector = vision.HandLandmarker.create_from_options(options)

def extract(path):
    img = cv2.imread(path)
    if img is None: return None
    img = cv2.resize(img, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
    rgb = np.ascontiguousarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    res = detector.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))
    if not res.hand_landmarks: return None
    return normalize([(p.x, p.y, p.z) for p in res.hand_landmarks[0]])

FIX = ['M', 'N']
TARGET, MAX_TRIES = 300, 2000

df = pd.read_csv(f'{BASE}/letters_baseline.csv', header=None)
df = df[~df[0].isin(FIX)]                      # drop the weak M and N rows

new_rows = []
for label in FIX:
    got = 0
    for f in sorted(os.listdir(os.path.join(ROOT, label)))[:MAX_TRIES]:
        v = extract(os.path.join(ROOT, label, f))
        if v is not None:
            new_rows.append([label, *v]); got += 1
            if got >= TARGET: break
    print(label, got)

df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)
df.to_csv(f'{BASE}/letters_baseline.csv', index=False, header=False)
print(df[0].value_counts().sort_index())

## 6. Quick baseline (random forest)
This checks the plumbing. Accuracy will look too good because the Kaggle images are very uniform. Do not trust it for the demo.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

df = pd.read_csv(f'{BASE}/letters_baseline.csv', header=None)
X, y = df.iloc[:, 1:].values.astype('float32'), df.iloc[:, 0].values
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)

rf = RandomForestClassifier(n_estimators=200, n_jobs=-1).fit(Xtr, ytr)
print(classification_report(yte, rf.predict(Xte)))

## 7. Train the browser-friendly MLP
Later, swap the CSV for the one from your teammate's browser recorder and split by person instead of randomly.

In [ ]:
import tensorflow as tf, json
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder().fit(y)
ytr_i, yte_i = le.transform(ytr), le.transform(yte)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(63,)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(len(le.classes_), activation='softmax'),
])
model.compile('adam', 'sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(Xtr, ytr_i, validation_data=(Xte, yte_i), epochs=40, batch_size=64)

with open(f'{BASE}/labels.json', 'w') as f:
    json.dump(list(le.classes_), f)
print(list(le.classes_))

## 8. Confusion matrix
Look for clashing letters (M/N/T, U/V). Collect more samples for those, or exclude them and say so in the pitch.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

pred = model.predict(Xte).argmax(1)
fig, ax = plt.subplots(figsize=(12, 12))
ConfusionMatrixDisplay.from_predictions(yte_i, pred, display_labels=le.classes_, xticks_rotation=90, ax=ax, colorbar=False)
plt.show()

## 9. Export for the browser (do this early!)
**Option A (most reliable): weights as JSON.** The model is just a few dense layers, so your frontend teammate can run it in about 15 lines of plain JS with no library. Use this if TF.js conversion gives you trouble.

In [ ]:
layers = []
for layer in model.layers:
    w = layer.get_weights()
    if len(w) == 2:   # Dense layers only (Dropout has no weights)
        layers.append({'W': w[0].tolist(), 'b': w[1].tolist()})

with open(f'{BASE}/mlp_weights.json', 'w') as f:
    json.dump({'layers': layers, 'labels': list(le.classes_)}, f)
print('saved', len(layers), 'dense layers to mlp_weights.json')

**JS forward pass to give your teammate** (relu on all layers except the last, softmax at the end):
```js
function predict(vec, net) {
  let x = vec;
  net.layers.forEach((L, i) => {
    const out = L.b.map((bias, j) => bias + x.reduce((s, v, k) => s + v * L.W[k][j], 0));
    x = i < net.layers.length - 1 ? out.map(v => Math.max(0, v)) : out;
  });
  const m = Math.max(...x), e = x.map(v => Math.exp(v - m)), s = e.reduce((a, b) => a + b);
  const p = e.map(v => v / s), idx = p.indexOf(Math.max(...p));
  return { label: net.labels[idx], confidence: p[idx] };
}
```
**Option B: TensorFlow.js** (try it, but don't fight it for more than 20 minutes).

In [ ]:
!pip install -q tensorflowjs
model.save('/content/model.h5')
!tensorflowjs_converter --input_format=keras /content/model.h5 {BASE}/web_model

## 10. Sanity check: does the JSON export match Keras?
Run this so the frontend handoff is trustworthy.

In [ ]:
def np_predict(vec):
    x = vec
    for i, L in enumerate(layers):
        x = x @ np.array(L['W']) + np.array(L['b'])
        if i < len(layers) - 1: x = np.maximum(0, x)
    e = np.exp(x - x.max()); p = e / e.sum()
    return le.classes_[p.argmax()], float(p.max())

for i in range(5):
    print(np_predict(Xte[i]), 'keras:', le.classes_[model.predict(Xte[i:i+1], verbose=0).argmax()], 'truth:', yte[i])

11. DO Tomorrow

In [ ]:
import glob
REC_DIR = f'{BASE}/recordings'
RAW = True

frames = []
for path in sorted(glob.glob(f'{REC_DIR}/*.csv')):
    d = pd.read_csv(path, header=None)
    d.columns = ['label'] + list(range(63))
    d['person'] = os.path.splitext(os.path.basename(path))[0]
    frames.append(d)
data = pd.concat(frames, ignore_index=True)

cols = list(range(63))
data[cols] = data[cols].apply(pd.to_numeric, errors='coerce')   # drops a header row if the recorder wrote one
data = data.dropna().reset_index(drop=True)

y = data['label'].values
people = data['person'].values
X = data[cols].values.astype('float32')
if RAW:
    X = np.array([normalize(r.reshape(21, 3)) for r in X], dtype='float32')

print(data.groupby('person').size())
print(pd.crosstab(data['label'], data['person']))   # look for letters with few samples

In [ ]:
import tensorflow as tf, matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import ConfusionMatrixDisplay

le = LabelEncoder().fit(y)
yi = le.transform(y)

def build(n_classes):
    m = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(63,)),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(n_classes, activation='softmax'),
    ])
    m.compile('adam', 'sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

def augment(X, copies=3, max_deg=10, scale=0.1, noise=0.01):
    out = []
    for _ in range(copies):
        pts = X.reshape(-1, 21, 3).copy()
        th = np.radians(np.random.uniform(-max_deg, max_deg, len(pts)))
        c, s = np.cos(th)[:, None], np.sin(th)[:, None]
        px, py = pts[:, :, 0].copy(), pts[:, :, 1].copy()
        pts[:, :, 0], pts[:, :, 1] = c * px - s * py, s * px + c * py
        pts *= np.random.uniform(1 - scale, 1 + scale, (len(pts), 1, 1))
        pts += np.random.normal(0, noise, pts.shape)
        out.append(pts.reshape(len(pts), 63))
    return np.vstack(out)

def lopo(use_aug):
    true_all, pred_all, conf_all = [], [], []
    for p in sorted(set(people)):
        te, tr = people == p, people != p
        Xt, yt = X[tr], yi[tr]
        if use_aug:
            Xt = np.vstack([Xt, augment(Xt)]); yt = np.concatenate([yt] + [yt] * 3)
        m = build(len(le.classes_))
        m.fit(Xt, yt, epochs=30, batch_size=64, verbose=0)
        probs = m.predict(X[te], verbose=0)
        pred = probs.argmax(1)
        print(f'  held out {p}: {(pred == yi[te]).mean():.3f}')
        true_all += list(yi[te]); pred_all += list(pred); conf_all += list(probs.max(1))
    return np.array(true_all), np.array(pred_all), np.array(conf_all)

for aug in [False, True]:
    print('augmentation:', aug)
    t, pr, cf = lopo(aug)
    print('  overall:', (t == pr).mean().round(3))

# details for the last run (with augmentation)
print('mean confidence when right:', cf[t == pr].mean().round(3), '| when wrong:', cf[t != pr].mean().round(3))
recall = pd.Series({le.classes_[k]: (pr[t == k] == k).mean() for k in range(len(le.classes_))}).sort_values()
print('weakest letters:\n', recall.head(8).round(2))
fig, ax = plt.subplots(figsize=(12, 12))
ConfusionMatrixDisplay.from_predictions(t, pr, display_labels=le.classes_, xticks_rotation=90, ax=ax, colorbar=False)
plt.show()

In [ ]:
#Train the final model
USE_AUG = True
Xf, yf = X, yi
if USE_AUG:
    Xf = np.vstack([X, augment(X)]); yf = np.concatenate([yi] * 4)

final = build(len(le.classes_))
final.fit(Xf, yf, epochs=30, batch_size=64, verbose=0)

layers = [{'W': l.get_weights()[0].tolist(), 'b': l.get_weights()[1].tolist()}
          for l in final.layers if len(l.get_weights()) == 2]
with open(f'{BASE}/mlp_weights.json', 'w') as f:
    json.dump({'layers': layers, 'labels': list(le.classes_)}, f)
print('saved mlp_weights.json with', len(layers), 'layers and', len(le.classes_), 'labels')